# Semana 11, Cierre Misión 2
Juan Esteban Camacho y Juan Diego Serpa

## Importación de librerías y dataset

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import PowerTransformer


In [52]:
df = pd.read_csv("S04_PD_Camacho_Serpa_DatasetImputacionInicial.csv")

## Etapa 1: Limpieza de inconsistencias y duplicados

### Quitamos espacios y capitalizamos las palabras

In [53]:
df["ciudad_tienda"] = df["ciudad_tienda"].str.strip().str.title()

In [54]:
df["categoria_producto"] = df["categoria_producto"].str.strip().str.title()

In [55]:
df["canal_compra"] = df["canal_compra"].str.strip().str.title()

In [56]:
df["metodo_pago"] = df["metodo_pago"].str.strip().str.title()

### Diccionarios de corrección

#### ciudad_tienda

In [57]:
# Diccionario de correcciones para ciudad_tienda
correcciones_ciudad = {
    "B/Manga": "Bucaramanga",
    "B/Quilla": "Barranquilla",
    "Baq": "Barranquilla",
    "Bga": "Bucaramanga",
    "Bog": "Bogotá",
    "Bogota": "Bogotá",
    "Bogota D.C.": "Bogotá",
    "Bogota Dc": "Bogotá",
    "Cartagena De Indias": "Cartagena",
    "Ctg": "Cartagena",
    "Cucuta": "Cúcuta",
    "Ibague": "Ibagué",
    "Manizales, Caldas": "Manizales",
    "Mde": "Medellín",
    "Medellin": "Medellín",
    "Medellin, Ant.": "Medellín",
    "Mzl": "Manizales",
    "Santamarta": "Santa Marta",
    "Santiago De Cali": "Cali",
    "Smr": "Santa Marta",
    "Sta. Marta": "Santa Marta"
}
df["ciudad_tienda"] = df["ciudad_tienda"].replace(correcciones_ciudad)

#### categoria_producto

In [58]:
correcciones_categoria = {
    "Electronica": "Electrónica",
    "Jugueteria": "Juguetería"
}
df["categoria_producto"] = df["categoria_producto"].replace(correcciones_categoria)

#### canal_compra y metodo_pago
No requieren correcciones

### Diccionario de meses en español

In [59]:
# 1. Normalizar texto
fechas_limpias = df["fecha_pedido"].astype(str).str.lower().str.strip()

# 2. Eliminar conectores comunes como "de"
fechas_limpias = fechas_limpias.str.replace(r'\bde\b', '', regex=True)

# 3. Mapeo completo (nombres largos primero para evitar reemplazos parciales)
meses_es = {
    'septiembre': '09', 'sept': '09', 'sep': '09',
    'noviembre': '11', 'nov': '11',
    'diciembre': '12', 'dic': '12',
    'febrero': '02', 'feb': '02',
    'agosto': '08', 'ago': '08',
    'enero': '01', 'ene': '01',
    'marzo': '03', 'mar': '03',
    'abril': '04', 'abr': '04',
    'mayo': '05', 'may': '05',
    'junio': '06', 'jun': '06',
    'julio': '07', 'jul': '07',
    'octubre': '10', 'oct': '10'
}

for mes_es, mes_num in meses_es.items():
    fechas_limpias = fechas_limpias.str.replace(rf'\b{mes_es}\b', mes_num, regex=True)

# 4. Limpiar espacios extra sobrantes
fechas_limpias = fechas_limpias.str.replace(r'\s+', ' ', regex=True).str.strip()

# 5. Convertir a datetime manteniendo el día primero cuando aplique
df["fecha_pedido"] = pd.to_datetime(fechas_limpias, format="mixed", dayfirst=True, errors="coerce")

### Eliminación de duplicados

In [60]:
# 1. Convertir la columna a datetime (asegurando el formato)
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], dayfirst=True, errors="coerce")

# 2. Ordenar situando las fechas nulas (NaT) al INICIO (na_position='first')
# Así, las fechas válidas más recientes quedarán al FINAL del DataFrame
df = df.sort_values("fecha_pedido", na_position="first")

# 3. Separar los registros que tienen correo de los que no lo tienen
# (para evitar que drop_duplicates borre masivamente las filas sin correo)
con_correo = df[df["correo_cliente"].notnull()].drop_duplicates(
    subset=["correo_cliente"], 
    keep="last" # Conserva la última fila, que ahora será la fecha válida más reciente
)

sin_correo = df[df["correo_cliente"].isnull()]

# 4. Volver a unir el dataset
df = pd.concat([con_correo, sin_correo]).sort_index()

df = df.drop_duplicates(keep="first")

## Etapa 2: Tratamiento de valores atípicos y ruido

### Cálculo del IQR

In [61]:
Q1 = df["monto_compra"].quantile(0.25)
Q3 = df["monto_compra"].quantile(0.75)
IQR = Q3 - Q1
print("Q1:", Q1, "Q3:", Q3, "Rango intercuartilico:", IQR)

Q1: 222700.0 Q3: 857400.0 Rango intercuartilico: 634700.0


In [62]:
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
print("Limite inferior:", limite_inferior)
print("Limite superior:", limite_superior)

Limite inferior: -729350.0
Limite superior: 1809450.0


In [63]:
outliers_iqr = df[(df["monto_compra"] < limite_inferior) | (df["monto_compra"] > limite_superior)]
print("Valores atipicos segun IQR:", len(outliers_iqr))

Valores atipicos segun IQR: 64816


### Z-Score

In [64]:
media = df["monto_compra"].mean()
desviacion = df["monto_compra"].std()

df["z_monto_compra"] = (df["monto_compra"] - media) / desviacion

outliers_zscore = df[df["z_monto_compra"].abs() > 3]
print("Valores atipicos segun z-score:", len(outliers_zscore))

Valores atipicos segun z-score: 123


### Tratamiento de outliers

#### venta_corporativa

In [65]:
df["es_venta_corporativa"] = df["monto_compra"] > limite_superior
df["es_venta_corporativa"].sum()

np.int64(53224)

#### monto_compra

In [66]:
df["monto_compra"] = df["monto_compra"].clip(upper=limite_superior, lower=0)

#### ventas_diarias

In [67]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce")
ventas_diarias = df.groupby("fecha_pedido")["monto_compra"].sum().reset_index()
ventas_diarias["suavizado"] = ventas_diarias["monto_compra"].rolling(window=7).mean()

#### precio_unitario

In [68]:
Q1_p = df["precio_unitario"].quantile(0.25)
Q3_p = df["precio_unitario"].quantile(0.75)
IQR_p = Q3_p - Q1_p
li_p = Q1_p - 1.5 * IQR_p
ls_p = Q3_p + 1.5 * IQR_p

outliers_precio = df[(df["precio_unitario"] < li_p) | (df["precio_unitario"] > ls_p)]

df["es_precio_alto"] = df["precio_unitario"] > ls_p
df["precio_unitario"] = df["precio_unitario"].clip(upper=ls_p, lower=0)
print("Registros marcados como precio alto:", df["es_precio_alto"].sum())
print("Media después del tratamiento:", df["precio_unitario"].mean())
print("Desv. estándar después:", df["precio_unitario"].std())
print("Máximo después:", df["precio_unitario"].max())

Registros marcados como precio alto: 6000
Media después del tratamiento: 340202.68964941567
Desv. estándar después: 211860.8547961304
Máximo después: 956450.0


#### unidades vendidas

In [69]:
df["unidades_vendidas"] = df["unidades_vendidas"].clip(lower=0, upper=100)
print("Media después:", df["unidades_vendidas"].mean())
print("Desv. estándar después:", df["unidades_vendidas"].std())
print("Mínimo:", df["unidades_vendidas"].min())
print("Máximo:", df["unidades_vendidas"].max())

Media después: 4.679110183639399
Desv. estándar después: 16.511494576472437
Mínimo: 0
Máximo: 100


#### edad_cliente

In [70]:
df["edad_cliente"] = df["edad_cliente"].clip(lower=0, upper=120)
print("Media después:", df["edad_cliente"].mean())
print("Desv. estándar después:", df["edad_cliente"].std())
print("Mínimo:", df["edad_cliente"].min())
print("Máximo:", df["edad_cliente"].max())

Media después: 47.28865776293823
Desv. estándar después: 21.96906962075148
Mínimo: 0
Máximo: 120


#### peso_pedido_kg

In [71]:
df["peso_pedido_kg"] = df["peso_pedido_kg"].clip(upper=100, lower=0)
print("Media después:", df["peso_pedido_kg"].mean())
print("Desv. estándar después:", df["peso_pedido_kg"].std())
print("Mínimo después:", df["peso_pedido_kg"].min())
print("Máximo después:", df["peso_pedido_kg"].max())

Media después: 15.92327078464107
Desv. estándar después: 19.386025915707275
Mínimo después: 8.0
Máximo después: 100.0


#### Resumen atípicos

In [72]:
resumen = pd.DataFrame({
    "min": df[["monto_compra", "precio_unitario", "unidades_vendidas", "edad_cliente", "peso_pedido_kg"]].min(),
    "max": df[["monto_compra", "precio_unitario", "unidades_vendidas", "edad_cliente", "peso_pedido_kg"]].max(),
    "media": df[["monto_compra", "precio_unitario", "unidades_vendidas", "edad_cliente", "peso_pedido_kg"]].mean(),
    "std": df[["monto_compra", "precio_unitario", "unidades_vendidas", "edad_cliente", "peso_pedido_kg"]].std(),
})
resumen

,min,max,media,std
monto_compra,0.0,1809450.0,624933.237229,539045.836870
precio_unitario,0.0,956450.0,340202.689649,211860.854796
unidades_vendidas,0.0,100.0,4.679110,16.511495
edad_cliente,0.0,120.0,47.288658,21.969070
peso_pedido_kg,8.0,100.0,15.923271,19.386026


## Etapa 3: Escalado de variables numéricas

### Entrenamiento

In [73]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)

### Escalado

In [74]:
escalador_edad = StandardScaler()
escalador_precio = MinMaxScaler()
escalador_monto = RobustScaler()
escalador_unidades = RobustScaler()

In [75]:
escalador_edad.fit(df_entrenamiento[["edad_cliente"]])
escalador_precio.fit(df_entrenamiento[['precio_unitario']])
escalador_monto.fit(df_entrenamiento[["monto_compra"]])
escalador_unidades.fit(df_entrenamiento[["unidades_vendidas"]])

RobustScaler()

In [76]:
df_entrenamiento["edad_cliente_escalada"] = escalador_edad.transform(df_entrenamiento[["edad_cliente"]])
df_prueba["edad_cliente_escalada"] = escalador_edad.transform(df_prueba[["edad_cliente"]])

In [77]:
df_entrenamiento["precio_unitario_escalada"] = escalador_precio.transform(df_entrenamiento[["precio_unitario"]])
df_prueba["precio_unitario_escalada"] = escalador_precio.transform(df_prueba[["precio_unitario"]])

In [78]:
df_entrenamiento["monto_compra_robusto"] = escalador_monto.transform(df_entrenamiento[["monto_compra"]])
df_prueba["monto_compra_robusto"] = escalador_monto.transform(df_prueba[["monto_compra"]])

In [79]:
df_entrenamiento["unidades_vendidas_robusto"] = escalador_unidades.transform(df_entrenamiento[["unidades_vendidas"]])
df_prueba["unidades_vendidas_robusto"] = escalador_unidades.transform(df_prueba[["unidades_vendidas"]])

### Unión de dataframes

In [80]:
df_final = pd.concat([df_entrenamiento, df_prueba])

## Etapa 4: Codificación de variables categóricas

### One-Hot Encoding

In [81]:
codificador_nominal = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
codificador_nominal.fit(df[["categoria_producto", "metodo_pago", "canal_compra"]])
# 1. Generar las variables dummy de forma aislada
dummies = pd.get_dummies(
    df[["categoria_producto", "metodo_pago", "canal_compra"]],
    prefix=["cat", "pago", "canal"],
    dtype=int
)

# 2. Concatenar las nuevas columnas binarias al DataFrame original
df = pd.concat([df, dummies], axis=1)

# (Opcional) Puedes revisar que la columna original sigue ahí
# print(df[["categoria_producto", "cat_Electrónica", "cat_Juguetería"]].head())

### Ordinal_Encoding

In [82]:
orden_lealtad = ["Bronce", "Plata", "Oro", "Platino"]
codificador_lealtad = OrdinalEncoder(categories=[orden_lealtad])
df["nivel_lealtad_cod"] = codificador_lealtad.fit_transform(df[["nivel_lealtad"]])
df[["nivel_lealtad", "nivel_lealtad_cod"]].drop_duplicates()

,nivel_lealtad,nivel_lealtad_cod
0,Platino,3.0
1,Bronce,0.0
2,Oro,2.0
4,Plata,1.0


### Separación de datasets

In [83]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)
df_entrenamiento = df_entrenamiento.dropna(subset=["monto_compra"])

#### Target Encoding ciudad_tienda

In [84]:
codificador_ciudad = TargetEncoder(cols=["ciudad_tienda"], smoothing=10)
codificador_ciudad.fit(df_entrenamiento["ciudad_tienda"], df_entrenamiento["monto_compra"])

TargetEncoder(cols=['ciudad_tienda'])

#### Transformar entrenamiento y prueba

In [85]:
df_entrenamiento["ciudad_tienda_cod"] = codificador_ciudad.transform(df_entrenamiento["ciudad_tienda"])
df_prueba["ciudad_tienda_cod"] = codificador_ciudad.transform(df_prueba["ciudad_tienda"])

### Agrupamiento de codigo_postal

In [86]:
conteo_postal = df_entrenamiento["codigo_postal"].value_counts()
postales_frecuentes = conteo_postal[conteo_postal >= 15].index

In [87]:
df_entrenamiento["codigo_postal_agrupado"] = df_entrenamiento["codigo_postal"].where(df_entrenamiento["codigo_postal"].isin(postales_frecuentes), "Otras")
df_prueba["codigo_postal_agrupado"] = df_prueba["codigo_postal"].where(df_prueba["codigo_postal"].isin(postales_frecuentes), "Otras")

### Unión de los conjuntos de entrenamiento y prueba

In [88]:
df = pd.concat([df_entrenamiento, df_prueba])

## Etapa 5: Transformaciones de fecha, texto y discretización

### Variables temporales extraídas

In [89]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])
df["anio_pedido"] = df["fecha_pedido"].dt.year
df["mes_pedido"] = df["fecha_pedido"].dt.month
df["trimestre_pedido"] = df["fecha_pedido"].dt.quarter
df["dia_semana_pedido"] = df["fecha_pedido"].dt.day_name()
df["es_fin_semana_pedido"] = df["fecha_pedido"].dt.dayofweek >= 5

### Variables de texto extraídas

In [90]:
df["comentario_cliente"] = df["comentario_cliente"].fillna("")
df["longitud_comentario"] = df["comentario_cliente"].str.len()
df["num_palabras_comentario"] = df["comentario_cliente"].str.split().str.len()
df["tiene_comentario"] = df["comentario_cliente"].str.len() > 0
df["menciona_demora"] = df["comentario_cliente"].str.contains(
    "tarde|demora|retraso|no llegó", case=False, na=False
)

### Separación en entrenamiento y prueba

In [91]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)

#### Discretización de edad_cliente

In [92]:
df["grupo_edad"] = pd.cut(df["edad_cliente"], bins=[17, 25, 35, 45, 60, 100], labels=["18 a 25", "26 a 35", "36 a 45", "46 a 60", "61 y más"])

### Separación en entrenamiento y prueba con columna edad_cliente

In [93]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)

#### Sesgo de monto_compra y transformación logarítmica

In [94]:
df["monto_compra"].skew()
df["monto_compra_log"] = np.log1p(df["monto_compra"])
df["monto_compra_log"].skew()

np.float64(-2.963324546265293)

#### Box-Cox

In [95]:
transformador_boxcox = PowerTransformer(method="box-cox")
transformador_boxcox.fit(df_entrenamiento[["monto_compra"]] + 1)

df_entrenamiento["monto_compra_boxcox"] = transformador_boxcox.transform(df_entrenamiento[["monto_compra"]] + 1)
df_prueba["monto_compra_boxcox"] = transformador_boxcox.transform(df_prueba[["monto_compra"]] + 1)

transformador_boxcox = PowerTransformer(method="box-cox")
transformador_boxcox.fit(df_entrenamiento[["monto_compra"]] + 1)

df_entrenamiento["monto_compra_boxcox"] = transformador_boxcox.transform(df_entrenamiento[["monto_compra"]] + 1)
df_prueba["monto_compra_boxcox"] = transformador_boxcox.transform(df_prueba[["monto_compra"]] + 1)

df_entrenamiento["monto_compra_boxcox"].skew()

np.float64(-0.5160025537590185)

### Unión de los conjuntos de entrenamiento y prueba

In [96]:
df_final = pd.concat([df_entrenamiento, df_prueba])

## Diccionario de datos

| Columna del dataset         | De qué semana viene | Qué significa                                                                                                                                           |
| --------------------------- | ------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `z_monto_compra`            | Semana 7            | Puntaje Z de `monto_compra`, que indica cuántas desviaciones estándar se encuentra una compra respecto a la media.                                      |
| `es_venta_corporativa`      | Semana 7            | Indica si `monto_compra` supera el límite superior calculado mediante IQR; se utiliza para identificar posibles ventas corporativas.                    |
| `es_precio_alto`            | Semana 7            | Indica si `precio_unitario` supera el límite superior calculado mediante IQR antes de aplicar el recorte del valor.                                     |
| `edad_cliente_escalada`     | Semana 8            | `edad_cliente` transformada mediante `StandardScaler`, conservando también la columna original.                                                         |
| `precio_unitario_escalada`  | Semana 8            | `precio_unitario` transformado mediante `MinMaxScaler`, conservando también la columna original.                                                        |
| `monto_compra_robusto`      | Semana 8            | `monto_compra` transformado mediante `RobustScaler`, conservando también la columna original.                                                           |
| `unidades_vendidas_robusto` | Semana 8            | `unidades_vendidas` transformada mediante `RobustScaler`, conservando también la columna original.                                                      |
| `nivel_lealtad_cod`         | Semana 9            | Nivel de lealtad codificado en orden real: Bronce=0, Plata=1, Oro=2, Platino=3.                                                                         |
| `ciudad_tienda_cod`         | Semana 9            | Ciudad de la tienda codificada mediante Target Encoding utilizando `monto_compra` como variable objetivo.                                               |
| `codigo_postal_agrupado`    | Semana 9            | Agrupa los códigos postales poco frecuentes bajo la categoría `Otras`; conserva individualmente los códigos con al menos 15 registros en entrenamiento. |
| `anio_pedido`               | Semana 10           | Año extraído de `fecha_pedido`.                                                                                                                         |
| `mes_pedido`                | Semana 10           | Mes numérico extraído de `fecha_pedido`.                                                                                                                |
| `trimestre_pedido`          | Semana 10           | Trimestre del año extraído de `fecha_pedido`.                                                                                                           |
| `dia_semana_pedido`         | Semana 10           | Día de la semana correspondiente a `fecha_pedido`.                                                                                                      |
| `es_fin_semana_pedido`      | Semana 10           | Indica si el pedido fue realizado en sábado o domingo.                                                                                                  |
| `longitud_comentario`       | Semana 10           | Número de caracteres presentes en `comentario_cliente`.                                                                                                 |
| `num_palabras_comentario`   | Semana 10           | Número de palabras presentes en `comentario_cliente`.                                                                                                   |
| `tiene_comentario`          | Semana 10           | Indica si el cliente dejó un comentario.                                                                                                                |
| `menciona_demora`           | Semana 10           | Indica si el comentario contiene términos relacionados con demora, retraso o entrega tardía.                                                            |
| `grupo_edad`                | Semana 10           | Edad del cliente discretizada en los grupos: 18 a 25, 26 a 35, 36 a 45, 46 a 60 y 61 y más.                                                             |
| `monto_compra_log`          | Semana 10           | Transformación logarítmica `log1p` de `monto_compra`, utilizada para reducir el sesgo de la variable.                                                   |
| `monto_compra_boxcox`       | Semana 10           | Transformación Box-Cox aplicada a `monto_compra + 1` para reducir su asimetría.                                                                         |


## Exportar dataset

## Verificación final del dataset

In [97]:
print("Filas del dataset final:", df_final.shape[0])
print("Columnas del dataset final:", df_final.shape[1])
print("Duplicados exactos:", df_final.duplicated().sum())

Filas del dataset final: 599000
Columnas del dataset final: 55
Duplicados exactos: 0


In [98]:
df_final.to_csv("S11_PD_Camacho_Serpa_DatasetTransformado.csv", index=False) 